In [1]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Inicializar Spark
spark = SparkSession.builder.appName("IntroStreaming").getOrCreate()


In [2]:
# 1. Crear un directorio temporal para simular la llegada de datos
directorio_stream = "datos_sensores"
os.makedirs(directorio_stream, exist_ok=True)

# 2. Definir el esquema (obligatorio en Streaming)
esquema = StructType([
    StructField("sensor", StringType(), True),
    StructField("temperatura", IntegerType(), True)
])

In [3]:
# 3. Leer el stream desde el directorio
stream_df = spark.readStream.schema(esquema).csv(directorio_stream)

# 4. Transformación: Calcular la temperatura máxima histórica por sensor
agregacion = stream_df.groupBy("sensor").max("temperatura")

# 5. Volcar los resultados en memoria (ideal para consultar desde la libreta)
query = (agregacion.writeStream
         .outputMode("complete")
         .format("memory")
         .queryName("tabla_resultados") # Nombre de la tabla temporal
         .start())

In [4]:
# --- SIMULACIÓN DE DATOS ENTRANTES EN TIEMPO REAL ---

# Lote 1: Llegan los primeros datos
with open(f"{directorio_stream}/lote1.csv", "w") as f:
    f.write("motor_principal,80\nventilador,40\n")

# Paso clave: con esta instrucción decimos a Spark que procese
# todo lo que encuentre en la carpeta donde entran datos en streaming antes de continuar.
query.processAllAvailable()

print("--- Resultados tras el Lote 1 ---")
spark.sql("SELECT * FROM tabla_resultados").show()

--- Resultados tras el Lote 1 ---
+---------------+----------------+
|         sensor|max(temperatura)|
+---------------+----------------+
|motor_principal|              80|
|     ventilador|              40|
+---------------+----------------+



In [5]:
# Lote 2: Llegan nuevos datos
with open(f"{directorio_stream}/lote2.csv", "w") as f:
    f.write("motor_principal,95\nventilador,42\nbomba_agua,50\n")

# Volvemos a forzar el procesamiento del nuevo lote
query.processAllAvailable()

print("--- Resultados tras el Lote 2 (Los máximos se actualizan solos) ---")
spark.sql("SELECT * FROM tabla_resultados").show()

# Detener el stream al terminar
query.stop()

--- Resultados tras el Lote 2 (Los máximos se actualizan solos) ---
+---------------+----------------+
|         sensor|max(temperatura)|
+---------------+----------------+
|motor_principal|              95|
|     bomba_agua|              50|
|     ventilador|              42|
+---------------+----------------+

